## Upload Stimulus Sequences to MongoDB

This notebook uploads the generated stimulus-sequence records to the MongoDB collection used by the experiment server.

Run it only when you intentionally want to replace or refresh the stimulus documents for a study. Before running, make sure MongoDB is reachable and that `pymongo` is installed in your Python environment.


### Connect to MongoDB

If MongoDB is running on a remote machine, first open an SSH tunnel from your laptop to the remote database port. This lets the notebook connect to MongoDB through `127.0.0.1:27017`.

You may need to restart the tunnel whenever your internet connection changes or the terminal reports `Broken pipe`. If MongoDB is already running locally on port `27017`, you can skip the SSH-tunnel cell.

In [1]:
# Uncomment if pymongo is not installed in your current environment.
# !pip install "pymongo<4.0"

In [2]:
import os
import pymongo as pm
import pandas as pd
import json
from IPython.display import clear_output

In [4]:
# Study and collection settings.
project_name = 'engaging-movies'
experiment_name = 'engaging-movies-cogsci'
mongo_collection_name = 'engaging-movies-cogsci'
iterationName = 'prolific0'

# Paths are resolved relative to the stimuli directory.
proj_dir = os.path.abspath('..')
stimuli_dir = os.getcwd()
output_dir = os.path.join(stimuli_dir, experiment_name)

# Set to False for a dry run that reads and validates the records without uploading.
write = True

In [ ]:
! ssh -fNL 27017:127.0.0.1:27017 <username>@<hostname>

bind [127.0.0.1]:27017: Address already in use
channel_setup_fwd_listener_tcpip: cannot listen to port: 27017
Could not request local forwarding.


In [ ]:
# Load local MongoDB credentials. The auth.json file is intentionally gitignored.
auth = pd.read_json(os.path.join(proj_dir, 'auth.json'), typ='series')
pswd = auth.password
user = auth.user
host = 'hostname'

# Connect through the local MongoDB port. If using the SSH tunnel above,
# this local port forwards to the remote MongoDB instance.
conn = pm.MongoClient('mongodb://sketchloop:' + pswd + '@127.0.0.1:27017')
db = conn['stimuli']
coll = db[mongo_collection_name]

### Load Stimulus Records

The JSON file contains one MongoDB document per participant assignment. Each record stores the participant identifier, condition assignments, player order, color order, and trial-level video metadata used by the experiment.

In [8]:
stimulus_record_path = os.path.join(stimuli_dir, 'stimuli_sequences_mongodb.json')

with open(stimulus_record_path, 'r') as f:
    json_string = f.read()
    data = json.loads(json_string)

type(data)
data[1].keys()

dict_keys(['participantID', 'numTrials', 'agentOrderCondition', 'trajectoryPair', 'colorOrderCondition', 'questionCondition', 'trials'])

In [16]:
J = data

In [19]:
num_existing_records = coll.estimated_document_count()
print(f'Existing records in {mongo_collection_name}: {num_existing_records}')

if write:
    # Replace the collection contents so MongoDB exactly matches this stimulus file.
    # Be careful: this deletes all existing records in the target collection.
    if num_existing_records > 0:
        db.drop_collection(mongo_collection_name)
        print('Dropped existing records from this collection.')

    for i, record in enumerate(J):
        coll.insert_one(record)
        clear_output(wait=True)
        print(f'Inserted {i + 1} of {len(J)} stimulus records.')

    print('Done inserting records into MongoDB!')
else:
    print('Dry run complete. Set write = True to upload records to MongoDB.')

Done inserting records into mongo!
